# Piloto 2026 — validación anidada del modelo de aromas

## tl;dr

**Veredicto pre-registrado:** `NO_VALID_MODEL`.  
**Modelo seleccionado:** `None`.

Se contrastan cuatro explicaciones anidadas con leave-one-reactor-run-out: equilibrio basal, respuesta biológica gradual después del pulso, reservorio de línea y ambos mecanismos. Un modelo sólo se acepta si mejora ≥30 % el NRMSE de condensado para ambos compuestos, no empeora >10 % el vino, mejora al menos la mitad de los reactores, conserva masa y no depende excesivamente de límites paramétricos.

## Contexto y supuestos

- El balance químico en vino y el condensado se modelan como observaciones distintas del mismo proceso.
- `rCO₂`, temperatura y pulsos se heredan del modelo piloto validado.
- La partición gas/líquido usa Morakul/Mouret y cambia con etanol y temperatura.
- La respuesta pospulso es gradual y específica por compuesto; no se interpreta automáticamente como cinética pura de nitrógeno porque el pulso y la transición térmica están confundidos en el protocolo A.
- El reservorio conserva masa, pero su constante sólo representa holdup físico si coincide con mediciones del tren de captura.
- Todos los seis reactores informaron el diseño: esta es validación interna retrospectiva, no confirmación prospectiva.
- 26211-P-12 se conserva en el primario y se excluye sólo en sensibilidad.

In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display, Image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'fermentation_model').exists():
    ROOT = ROOT.parent
if not (ROOT / 'fermentation_model').exists():
    raise RuntimeError('Execute from the repository or a descendant directory')
sys.path.insert(0, str(ROOT / 'fermentation_model'))
from pilot_2026 import run_aroma_nested_model_validation_2026 as analysis
result = analysis.load_results() if os.environ.get('PILOT_AROMA_REUSE_RESULTS') == '1' else analysis.run_analysis()
print('Resultados:', analysis.RESULTS_DIR.relative_to(ROOT))
print('Veredicto:', result['gate']['verdict'])
print('Modelo seleccionado:', result['gate']['selected_model'])

## Calidad de datos

In [ ]:
display(result['anomaly_summary'].head(12).round(3))
display(Image(filename=analysis.FIGURE_DIR / '01_sample_anomaly_audit.png'))

## Forzantes del proceso

In [ ]:
display(result['pulses'][['batch', 'pulse_time_h', 'timing_source']].round(3))
display(Image(filename=analysis.FIGURE_DIR / '03_rco2_temperature_pulse_drivers.png'))

## Resultados de validación cruzada

In [ ]:
display(result['metrics'].round(4))
display(result['comparison'].round(4))
display(Image(filename=analysis.FIGURE_DIR / '02_loro_nrmse_comparison.png'))

## Curvas ajustadas y pérdida por intervalo

In [ ]:
for species in analysis.SPECIES_LABELS:
    display(Image(filename=analysis.FIGURE_DIR / f'04_liquid_{species}.png'))
    display(Image(filename=analysis.FIGURE_DIR / f'05_condensate_{species}.png'))

## Estabilidad e identificabilidad práctica

In [ ]:
display(result['stability'].round(4))
display(result['parameters'].round(5))
display(result['fit_validation'].round(5))
display(Image(filename=analysis.FIGURE_DIR / '06_parameter_stability.png'))

## Takeaways

- `PASS` significa que una extensión cumplió todos los guardrails internos para ambos compuestos; no equivale a validación prospectiva.
- `NO_VALID_MODEL` significa que la evidencia actual no separa de forma transferible producción, transferencia y captura bajo los criterios definidos.
- Una respuesta pospulso útil sólo para acetato de isoamilo es biológicamente plausible, pero no satisface un modelo común si octanoato no cruza el mismo gate.
- Un reservorio con τ de decenas de horas debe tratarse como término fenomenológico hasta medir holdup, eficiencia del condensador y aroma en gas.
- El siguiente experimento discriminante es medir simultáneamente vino, gas de salida y condensado alrededor del pulso.

In [ ]:
assert result['gate']['verdict'] in {'PASS', 'NO_VALID_MODEL'}
assert len(result['figures']) == 8
assert result['fit_validation']['maximum_relative_mass_balance_error'].max() <= 1e-8
print('Notebook ejecutado sin errores.')
print('Figuras embebidas:', len(result['figures']))